# 5量子ビット Deutsch-Jozsa アルゴリズム
## 複合誤り緩和手法（ZNE + 測定誤り緩和）の有効性検証

**問題設定：** 未知のブラックボックス関数 $f: \{0,1\}^4 \to \{0,1\}$ が
「定数関数（Constant）」か「バランス関数（Balanced）」かを、
最小クエリ数（1回）で判定する。

- 入力: $n=4$ 量子ビット + アンシラ 1 量子ビット = **合計 5 量子ビット**
- オラクル: パリティ関数 $f(x) = x_0 \oplus x_1 \oplus x_2 \oplus x_3$（バランス関数）
- バックエンド: `FakeManilaV2`（IBM 5量子ビット実機のノイズプロファイル）
- 誤り緩和: **測定誤り緩和（Readout Error Mitigation）** + **ゼロノイズ外挿法（ZNE）**

---
## 0. 環境セットアップ

In [ ]:
# 必要ライブラリのインストール（初回のみ）
# !pip install qiskit qiskit-aer qiskit-ibm-runtime mitiq matplotlib numpy

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import Counter

from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.visualization import plot_histogram
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime.fake_provider import FakeManilaV2
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

matplotlib.rcParams['figure.dpi'] = 150

# 実験パラメータ
N      = 4       # 入力量子ビット数
SHOTS  = 10000   # ショット数
SEED   = 42

print(f'入力量子ビット n={N}, アンシラ 1bit → 合計 {N+1} qubit')
print(f'ショット数: {SHOTS}')

---
## 1. アルゴリズムの概要

### 回路の流れ

$$
|0\rangle^{\otimes 4}|1\rangle
\xrightarrow{H^{\otimes 5}}
\frac{1}{4}\sum_{x \in \{0,1\}^4}|x\rangle|-\rangle
\xrightarrow{U_f}
\frac{1}{4}\sum_{x}(-1)^{f(x)}|x\rangle|-\rangle
\xrightarrow{H^{\otimes 4}}
\text{測定}
$$

- **バランス関数** $f(x)=x_0\oplus x_1\oplus x_2\oplus x_3$（パリティ）の場合、
  位相キックバックにより $|0000\rangle$ の振幅がゼロになり、
  $|1111\rangle$ が確定的に観測される（本実装のオラクル設計）。
- **定数関数** の場合は $|0000\rangle$ が確定的に観測される。

| | 古典（決定論） | 量子 |
|---|---|---|
| 必要クエリ数 | $2^{n-1}+1 = 9$ 回 | **1 回** |

---
## 2. オラクルと回路の実装

**引用:** 回路の基本構成は Qiskit Textbook "Deutsch-Jozsa Algorithm"
(https://qiskit.org/textbook/) の解説コードを参考に、
$n=4$ のパリティ検証オラクルへ独自に拡張した。

In [ ]:
def oracle_parity(qc: QuantumCircuit, x_reg: QuantumRegister, anc: QuantumRegister):
    """
    バランス関数オラクル: f(x) = x0 ⊕ x1 ⊕ x2 ⊕ x3 （パリティ関数）
    全入力量子ビットから CNOT でアンシラへ位相キックバック。
    理論的観測結果: |1111⟩ が 100%

    参考: Qiskit Textbook Deutsch-Jozsa の balanced oracle 実装を基に拡張
    """
    for q in x_reg:
        qc.cx(q, anc[0])


def oracle_constant_0(qc, x_reg, anc):
    """定数関数 f(x)=0: 恒等変換（操作なし）。観測結果: |0000⟩ が 100%"""
    pass


def dj_circuit(oracle_fn, n: int = N, name: str = "DJ") -> QuantumCircuit:
    """
    Deutsch-Jozsa 量子回路を生成する。

    Parameters
    ----------
    oracle_fn : callable
        オラクル関数 (qc, x_reg, anc_reg) を引数に取る
    n : int
        入力量子ビット数（本実験では n=4）
    """
    x   = QuantumRegister(n, 'x')
    anc = QuantumRegister(1, 'anc')
    c   = ClassicalRegister(n, 'c')
    qc  = QuantumCircuit(x, anc, c, name=name)

    # Step 1: アンシラを |1⟩ に初期化
    qc.x(anc)
    qc.barrier(label='init')

    # Step 2: 全量子ビットにアダマール変換 H^⊗(n+1)
    qc.h(x)
    qc.h(anc)
    qc.barrier(label='H⊗(n+1)')

    # Step 3: オラクル U_f の適用
    oracle_fn(qc, x, anc)
    qc.barrier(label='U_f')

    # Step 4: 入力レジスタに H^⊗n を再適用
    qc.h(x)
    qc.barrier(label='H⊗n')

    # Step 5: 入力レジスタのみ測定（アンシラは測定しない）
    qc.measure(x, c)
    return qc


# 回路の確認
qc_balanced = dj_circuit(oracle_parity, name='DJ-Balanced(Parity)')
print(qc_balanced.draw(output='text', fold=100))
print(f'\ndepth={qc_balanced.depth()}, gate count={qc_balanced.size()}')

In [ ]:
# 回路図を描画
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
dj_circuit(oracle_parity,     name='Balanced (Parity)').draw(output='mpl', ax=axes[0], style='iqp')
dj_circuit(oracle_constant_0, name='Constant-0').draw(       output='mpl', ax=axes[1], style='iqp')
axes[0].set_title('バランス関数オラクル（パリティ）', fontsize=11, fontweight='bold')
axes[1].set_title('定数関数オラクル（f=0）',           fontsize=11, fontweight='bold')
plt.suptitle('Deutsch-Jozsa 5量子ビット回路', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('circuit.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3. バックエンドの準備

In [ ]:
backend   = FakeManilaV2()
noisy_sim = AerSimulator.from_backend(backend)
pm        = generate_preset_pass_manager(backend=backend, optimization_level=1)

print(f'Backend : {backend.name}')
print(f'Qubits  : {backend.num_qubits}')
print(f'Gates   : {backend.operation_names}')

---
## 4. 実験 A: 理想シミュレータ（ノイズなし）

In [ ]:
ideal_sim = AerSimulator()   # ノイズなし

qc_ideal = dj_circuit(oracle_parity)
counts_ideal = ideal_sim.run(qc_ideal, shots=SHOTS, seed_simulator=SEED).result().get_counts()

print('=== A: 理想環境（Ideal）===')
print(counts_ideal)
p_1111_ideal = counts_ideal.get('1111', 0) / SHOTS
p_0000_ideal = counts_ideal.get('0000', 0) / SHOTS
print(f'P(|1111⟩) = {p_1111_ideal:.4f}')
print(f'P(|0000⟩) = {p_0000_ideal:.4f}')

---
## 5. 実験 B: Fake Backend（誤り緩和なし）

In [ ]:
qc_noisy  = pm.run(dj_circuit(oracle_parity))
counts_noisy = noisy_sim.run(qc_noisy, shots=SHOTS, seed_simulator=SEED).result().get_counts()

print('=== B: Fake Backend（緩和なし）===')
top5 = sorted(counts_noisy.items(), key=lambda x: -x[1])[:5]
for k, v in top5:
    print(f'  |{k}⟩: {v} ({v/SHOTS*100:.1f}%)')

total_noisy   = sum(counts_noisy.values())
p_1111_noisy  = counts_noisy.get('1111', 0) / total_noisy
p_0000_noisy  = counts_noisy.get('0000', 0) / total_noisy
print(f'\nP(|1111⟩) = {p_1111_noisy:.4f}')
print(f'P(|0000⟩) = {p_0000_noisy:.4f}')

---
## 6. 誤り緩和の実装

### 6.1 測定誤り緩和（Readout Error Mitigation）

較正行列 $A$（$A_{ij}$ = 状態 $j$ を準備したとき状態 $i$ が測定される確率）を構築し、  
擬似逆行列 $A^+$ を測定結果ベクトルに乗算して補正する。

$$\hat{p} = A^+ \cdot \tilde{p}$$

**引用:** 較正行列の構成アプローチは Qiskit Runtime Primitives の公式ドキュメント
（`resilience_level` 関連設定）を参考に、本実験向けに独自実装した。

### 6.2 ゼロノイズ外挿法（ZNE: Zero Noise Extrapolation）

CNOT ゲートをノイズスケールファクター $\lambda = 1, 3, 5$ 倍に折り畳み（gate-folding）、  
各スケールで期待値を測定してリチャードソン外挿によりゼロノイズ極限 $\lambda \to 0$ を推定する。

**引用:** ZNE の外挿ロジックは Mitiq (Unitary Fund) ライブラリの `mitiq.zne` モジュール
（LaRose et al., 2022, *Quantum*, 6, 774）を参考に、Qiskit 回路向けに独自実装した。

In [ ]:
# ===== 6.1 測定誤り緩和 =====
# 参考: Qiskit Runtime Primitives 公式ドキュメント（resilience_level 関連）を基に独自実装

def build_calibration_matrix(simulator, n: int, shots: int, pass_manager=None) -> np.ndarray:
    """
    n-qubit 測定誤り較正行列 A を構築する。
    2^n 個の計算基底状態それぞれを準備・測定し、行列要素を推定する。
    A[i, j] = 状態 j を準備したとき、状態 i が測定される確率
    """
    dim = 2 ** n
    A   = np.zeros((dim, dim))

    for j in range(dim):
        qc = QuantumCircuit(n, n)
        # ビット j を 2 進数展開してフリップ
        for bit in range(n):
            if (j >> bit) & 1:
                qc.x(bit)
        qc.measure(range(n), range(n))

        if pass_manager is not None:
            qc = pass_manager.run(qc)

        counts = simulator.run(qc, shots=shots, seed_simulator=SEED).result().get_counts()
        for bitstr, cnt in counts.items():
            i = int(bitstr.replace(' ', ''), 2)
            if i < dim:
                A[i, j] += cnt / shots
    return A


def apply_readout_mitigation(counts: dict, A_inv: np.ndarray, n: int, shots: int) -> dict:
    """
    測定誤り緩和を適用し、補正済み確率分布 (counts 形式) を返す。
    """
    dim   = 2 ** n
    p_raw = np.zeros(dim)
    total = sum(counts.values())

    for bitstr, cnt in counts.items():
        idx = int(bitstr.replace(' ', ''), 2)
        if idx < dim:
            p_raw[idx] += cnt / total

    p_mit = A_inv @ p_raw
    p_mit = np.clip(p_mit, 0, None)
    if p_mit.sum() > 0:
        p_mit /= p_mit.sum()

    return {format(i, f'0{n}b'): int(round(p_mit[i] * shots))
            for i in range(dim) if p_mit[i] > 0}


print('較正行列を構築中（2^4=16 回路実行）...')
A_cal = build_calibration_matrix(noisy_sim, N, shots=SHOTS, pass_manager=pm)
A_inv = np.linalg.pinv(A_cal)

diag = np.diag(A_cal)
print(f'対角成分（正測定率） 平均: {diag.mean():.4f}')
print(f'推定測定誤り率:            {(1 - diag.mean())*100:.2f}%')

In [ ]:
# ===== 6.2 ZNE: Gate Folding によるノイズスケール増幅 =====
# 参考: Mitiq (Unitary Fund) mitiq.zne モジュールの外挿ロジックを参考に独自実装
# LaRose et al. (2022). Mitiq: A software package for error mitigation on noisy
# quantum computers. Quantum, 6, 774.

def fold_gates(qc: QuantumCircuit, scale: int) -> QuantumCircuit:
    """
    Gate-folding によりノイズを scale 倍に増幅した回路を返す。
    各ゲートを G → G G† G（scale=3）のように奇数倍折り畳むことで
    論理的等価性を保ちながらノイズを増幅する。

    Parameters
    ----------
    scale : int (奇数)
        ノイズスケールファクター（1, 3, 5 ...）
    """
    assert scale % 2 == 1 and scale >= 1, 'scale は正の奇数である必要があります'
    if scale == 1:
        return qc.copy()

    n_fold = (scale - 1) // 2  # 追加する G†G ペア数
    folded = QuantumCircuit(*qc.qregs, *qc.cregs)

    for instr, qargs, cargs in qc.data:
        folded.append(instr, qargs, cargs)
        for _ in range(n_fold):
            # G† を追加
            folded.append(instr.inverse(), qargs, cargs)
            # G を追加
            folded.append(instr, qargs, cargs)

    return folded


def richardson_extrapolation(scales: list, expectations: list, target: float = 0.0) -> float:
    """
    リチャードソン外挿（多項式フィット）によりゼロノイズ極限の期待値を推定する。

    Parameters
    ----------
    scales       : ノイズスケールファクターのリスト [1, 3, 5, ...]
    expectations : 各スケールで測定した期待値のリスト
    target       : 外挿するノイズレベル（デフォルト 0: ゼロノイズ）
    """
    coeffs = np.polyfit(scales, expectations, deg=len(scales) - 1)
    poly   = np.poly1d(coeffs)
    return float(poly(target))


# ZNE の実行
SCALE_FACTORS = [1, 3, 5]   # ノイズスケールファクター
TARGET_KEY    = '1111'      # バランス関数の正解状態

print('ZNE: ノイズスケール別に回路を実行中...')
zne_data = {}   # {scale: (counts, p_1111)}

for scale in SCALE_FACTORS:
    qc_folded = fold_gates(dj_circuit(oracle_parity), scale=scale)
    qc_t      = pm.run(qc_folded)
    counts    = noisy_sim.run(qc_t, shots=SHOTS, seed_simulator=SEED).result().get_counts()
    total     = sum(counts.values())
    p_target  = counts.get(TARGET_KEY, 0) / total
    zne_data[scale] = (counts, p_target)
    print(f'  scale={scale}: P(|{TARGET_KEY}⟩) = {p_target:.4f}')

# ZNE 外挿
scales_list = list(zne_data.keys())
p_list      = [zne_data[s][1] for s in scales_list]
p_zne       = richardson_extrapolation(scales_list, p_list, target=0.0)
p_zne       = float(np.clip(p_zne, 0.0, 1.0))
print(f'\nZNE 外挿結果 P(|{TARGET_KEY}⟩) → {p_zne:.4f}')

In [ ]:
# ===== 複合緩和（REM → ZNE）=====
# scale=1（緩和なしと同じ回路）の counts に測定誤り緩和を適用してから ZNE に使う

print('複合誤り緩和（REM + ZNE）を計算中...')
rem_expectations = []

for scale in SCALE_FACTORS:
    raw_counts = zne_data[scale][0]
    # 測定誤り緩和
    mit_counts = apply_readout_mitigation(raw_counts, A_inv, N, SHOTS)
    total_mit  = sum(mit_counts.values())
    p_mit_target = mit_counts.get(TARGET_KEY, 0) / max(total_mit, 1)
    rem_expectations.append(p_mit_target)
    print(f'  scale={scale}: P(|{TARGET_KEY}⟩) after REM = {p_mit_target:.4f}')

# ZNE 外挿（REM 適用後）
p_combined = richardson_extrapolation(SCALE_FACTORS, rem_expectations, target=0.0)
p_combined = float(np.clip(p_combined, 0.0, 1.0))
print(f'\n複合緩和（REM+ZNE）外挿結果 P(|{TARGET_KEY}⟩) → {p_combined:.4f}')

# 測定誤り緩和のみ（scale=1）の結果
counts_rem_only = apply_readout_mitigation(zne_data[1][0], A_inv, N, SHOTS)
total_rem       = sum(counts_rem_only.values())
p_rem_only      = counts_rem_only.get(TARGET_KEY, 0) / max(total_rem, 1)
p_0000_rem_only = counts_rem_only.get('0000', 0)  / max(total_rem, 1)

# 複合緩和の |0000⟩ 確率（外挿なし scale=1 の REM 値で代用）
p_0000_combined = p_0000_rem_only  # ZNE は P(1111) に対して外挿; |0000| は REM で十分小さい

---
## 7. 結果の集計と可視化

In [ ]:
# ===== 数値サマリー =====
p_0000_noisy    = counts_noisy.get('0000', 0) / sum(counts_noisy.values())
p_0000_combined = counts_rem_only.get('0000', 0) / max(sum(counts_rem_only.values()), 1)

summary = {
    '(A) 理想環境':           {'1111': p_1111_ideal,  '0000': p_0000_ideal},
    '(B) 緩和なし':           {'1111': p_1111_noisy,  '0000': p_0000_noisy},
    '(C) 複合緩和 (REM+ZNE)': {'1111': p_combined,    '0000': p_0000_combined},
}

print('=== 実験結果サマリー ===')
print(f'{"条件":<26} {"P(|1111⟩) 正解":>16} {"P(|0000⟩) 誤答":>16}')
print('-' * 60)
for label, vals in summary.items():
    print(f'{label:<26} {vals["1111"]:>16.4f} {vals["0000"]:>16.4f}')

In [ ]:
# ===== メインの比較グラフ（レポート 4. 実験結果 に貼付用）=====

labels      = ['(A) 理想環境', '(B) 緩和なし', '(C) 複合緩和\n(REM+ZNE)']
vals_1111   = [summary[k]['1111'] for k in summary]
vals_0000   = [summary[k]['0000'] for k in summary]

x     = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))

bars1 = ax.bar(x - width/2, vals_1111, width,
               label='|1111⟩ 正解状態（バランス関数）',
               color=['#2166ac', '#d6604d', '#4dac26'], alpha=0.88, edgecolor='white', linewidth=0.8)
bars2 = ax.bar(x + width/2, vals_0000, width,
               label='|0000⟩ 致命的誤答（定数関数と混同）',
               color=['#9ecae1', '#fdae6b', '#a1d99b'], alpha=0.88, edgecolor='white', linewidth=0.8,
               hatch='///')

for bar in bars1:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.01,
            f'{h*100:.1f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')
for bar in bars2:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.01,
            f'{h*100:.1f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=12)
ax.set_ylabel('観測確率', fontsize=12)
ax.set_ylim(0, 1.18)
ax.set_title(
    'Deutsch-Jozsa アルゴリズム（バランス関数 / パリティオラクル）\n'
    '3条件における正解・誤答状態の観測確率比較',
    fontsize=13, fontweight='bold'
)
ax.legend(fontsize=11, loc='upper right')
ax.grid(axis='y', alpha=0.3)
ax.axhline(1.0, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)

plt.tight_layout()
plt.savefig('result_comparison.png', dpi=150, bbox_inches='tight')
print('グラフを result_comparison.png に保存しました')
plt.show()

In [ ]:
# ===== ZNE 外挿グラフ =====

# 補間用カーブ
x_fit  = np.linspace(0, 5.5, 300)
coeffs = np.polyfit(SCALE_FACTORS, rem_expectations, deg=2)
y_fit  = np.poly1d(coeffs)(x_fit)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(x_fit, y_fit, 'k--', linewidth=1.5, alpha=0.6, label='外挿曲線（2次多項式）')
ax.scatter(SCALE_FACTORS, rem_expectations,
           s=100, zorder=5, color='#d6604d', label='測定点（REM適用後）')
ax.scatter([0], [p_combined],
           s=160, zorder=6, color='#4dac26', marker='*',
           label=f'ZNE外挿値（λ=0）: {p_combined:.4f}')
ax.scatter([1], [p_1111_noisy],
           s=80, zorder=4, color='gray', marker='x', label=f'緩和なし（λ=1）: {p_1111_noisy:.4f}')

for s, p in zip(SCALE_FACTORS, rem_expectations):
    ax.annotate(f'{p:.4f}', (s, p), textcoords='offset points',
                xytext=(8, 6), fontsize=9)

ax.set_xlabel('ノイズスケールファクター λ', fontsize=12)
ax.set_ylabel('P(|1111⟩)', fontsize=12)
ax.set_title('ZNE: ゼロノイズ外挿法によるP(|1111⟩)の推定', fontsize=12, fontweight='bold')
ax.set_xlim(-0.3, 5.8)
ax.axvline(0, color='green', linestyle=':', linewidth=1, alpha=0.5)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('zne_extrapolation.png', dpi=150, bbox_inches='tight')
print('ZNE グラフを zne_extrapolation.png に保存しました')
plt.show()

In [ ]:
# ===== 測定分布の全量ヒストグラム（3条件並べて比較）=====

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

datasets = [
    (counts_ideal,     '(A) 理想環境（Ideal）',           '#2166ac'),
    (counts_noisy,     '(B) 緩和なし（Unmitigated）',      '#d6604d'),
    (counts_rem_only,  '(C) 複合緩和 REM+ZNE',             '#4dac26'),
]

for ax, (counts, title, color) in zip(axes, datasets):
    plot_histogram(counts, ax=ax, color=[color], title=title)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.tick_params(axis='x', rotation=45)

plt.suptitle('測定確率分布の比較（バランス関数 / パリティオラクル）',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('histogram_all.png', dpi=150, bbox_inches='tight')
print('ヒストグラムを histogram_all.png に保存しました')
plt.show()

---
## 8. 誤り行列の可視化

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, mat, title in zip(axes,
                           [A_cal, A_inv],
                           ['較正行列 A', '擬似逆行列 A⁺']):
    im = ax.imshow(mat, cmap='Blues', aspect='auto', vmin=0)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('準備した状態 j', fontsize=10)
    ax.set_ylabel('測定された状態 i', fontsize=10)
    fig.colorbar(im, ax=ax)

plt.suptitle('4-qubit 測定誤り較正行列（FakeManilaV2）',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('calibration_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 9. 最終サマリー

In [ ]:
print('=' * 70)
print('表1: 各実行条件における状態の観測確率（10,000 ショット）')
print('=' * 70)
print(f'{"実行条件":<28} {"正解 |1111⟩":>14} {"誤答 |0000⟩":>14}')
print('-' * 70)
rows = [
    ('(A) 理想環境',          p_1111_ideal,  p_0000_ideal),
    ('(B) 緩和なし',          p_1111_noisy,  p_0000_noisy),
    ('(C) 複合緩和(REM+ZNE)', p_combined,    p_0000_combined),
]
for label, p1, p0 in rows:
    print(f'{label:<28} {p1*100:>13.1f}% {p0*100:>13.1f}%')
print('=' * 70)

print(f'\n誤り緩和による改善:')
print(f'  P(|1111⟩): {p_1111_noisy*100:.1f}% → {p_combined*100:.1f}% '
      f'（+{(p_combined - p_1111_noisy)*100:.1f}pp）')
print(f'  P(|0000⟩): {p_0000_noisy*100:.1f}% → {p_0000_combined*100:.1f}% '
      f'（{(p_0000_combined - p_0000_noisy)*100:+.1f}pp）')

---
## 参考文献

1. Deutsch, D., & Jozsa, R. (1992). *Rapid solution of problems by quantum computation.*  
   Proceedings of the Royal Society of London. Series A, **439**(1907), 553–558.
2. Qiskit Textbook: *Deutsch-Jozsa Algorithm* (https://qiskit.org/textbook/)  
   — オラクル・回路構成の基本実装を参考に n=4 向けに拡張。
3. LaRose, R., et al. (2022). *Mitiq: A software package for error mitigation on noisy quantum computers.*  
   Quantum, **6**, 774.  
   — ZNE の gate-folding・外挿ロジックを参考に Qiskit 向けに独自実装。
4. Qiskit Runtime Primitives 公式ドキュメント（resilience_level 関連設定）  
   — 測定誤り緩和の較正行列構成アプローチを参考に独自実装。